# Challenge 10 Colab Ultra: signal feature engineering

Esta variante explota la hipotesis mas importante que aun no habiamos incorporado al pipeline:

- los predictores `V1 ... V200` muy probablemente representan una señal ordenada
- si eso es cierto, el mejor salto puede venir de `feature engineering` de señal y no de otro algoritmo

## Estrategia

1. Construir bancos de features en dominio temporal y frecuencial.
2. Evaluar candidatos `KNN` y `SVM` sobre esas features.
3. Probar tambien una version hibrida que concatena features ingenierizadas con `raw PCA`.
4. Exportar la mejor variante para submission y stacking.

## 0. Imports and global constants

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import gc
import itertools
import json
import platform
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import ParameterSampler, StratifiedKFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import SVC

try:
    import psutil
except ImportError:
    psutil = None

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["savefig.bbox"] = "tight"

RANDOM_STATE = 301655
VALID_SIZE = 0.20
NOTEBOOK_SLUG = "challenge_10_signal_features_colab_ultra"

## 1. Create the Colab workspace

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files  # type: ignore
else:
    files = None

if IN_COLAB:
    WORKSPACE_ROOT = Path("/content/challenge_signal_features_ultra_workspace")
else:
    cwd = Path.cwd().resolve()
    if (cwd / "challenge" / "data" / "training.csv").exists():
        WORKSPACE_ROOT = cwd / "challenge"
    elif (cwd / "data" / "training.csv").exists():
        WORKSPACE_ROOT = cwd
    else:
        WORKSPACE_ROOT = cwd / "challenge_signal_features_ultra_workspace"

DATA_DIR = WORKSPACE_ROOT / "data"
TRAIN_PATH = DATA_DIR / "training.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "sample.csv"

OUTPUT_ROOT = WORKSPACE_ROOT / "output"
PERSIST_ROOT = OUTPUT_ROOT / NOTEBOOK_SLUG
CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints"
SUBMISSION_DIR = WORKSPACE_ROOT / "submissions"
EXPORT_DIR = WORKSPACE_ROOT / "exports"

for path in [WORKSPACE_ROOT, DATA_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("PERSIST_ROOT:", PERSIST_ROOT)

## 2. Inspect the workspace and expected files

In [ ]:
expected_files = {
    "training.csv": TRAIN_PATH,
    "test.csv": TEST_PATH,
    "sample.csv": SAMPLE_PATH,
}

print("Workspace directories:")
for path in [WORKSPACE_ROOT, DATA_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    print("-", path)

print("\nData file status:")
for filename, path in expected_files.items():
    print(f"- {filename}: {'OK' if path.exists() else 'MISSING'} -> {path}")

## 3. Optional: upload the CSV files manually

In [ ]:
UPLOAD_DATA_FILES = False

if UPLOAD_DATA_FILES:
    if not IN_COLAB:
        raise RuntimeError("This upload helper is intended for Google Colab.")

    uploaded = files.upload()
    for original_name, file_bytes in uploaded.items():
        filename = Path(original_name).name
        target_path = DATA_DIR / filename
        target_path.write_bytes(file_bytes)
        print("Saved:", target_path)
else:
    print("Set UPLOAD_DATA_FILES = True if you want to upload training.csv, test.csv and sample.csv.")

## 4. Optional: restore resume bundles

In [ ]:
RESTORE_RESUME_BUNDLE = False

if RESTORE_RESUME_BUNDLE:
    if not IN_COLAB:
        raise RuntimeError("This restore helper is intended for Google Colab.")

    uploaded = files.upload()
    zip_names = [Path(name).name for name in uploaded if str(name).lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one ZIP file when restoring a resume bundle.")

    bundle_name = zip_names[0]
    bundle_path = EXPORT_DIR / bundle_name
    bundle_path.write_bytes(uploaded[bundle_name])
    with zipfile.ZipFile(bundle_path, "r") as zip_file:
        zip_file.extractall(WORKSPACE_ROOT)
    print("Resume bundle restored into:", WORKSPACE_ROOT)
else:
    print("Set RESTORE_RESUME_BUNDLE = True if you want to restore a previous ZIP bundle.")

## 5. Checkpoint helpers and reusable utilities

In [ ]:
DEFAULT_BUNDLE_NAME = "challenge_10_signal_features_colab_ultra_resume.zip"


def save_current_figure(filename: str) -> Path:
    path = PERSIST_ROOT / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    return path


def write_json_atomic(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, indent=2))
    tmp_path.replace(path)


def save_dataframe_atomic(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def read_dataframe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def normalize_value(value):
    if isinstance(value, np.generic):
        return value.item()
    return value


def candidate_signature(params: dict) -> str:
    normalized = {key: normalize_value(value) for key, value in params.items()}
    return json.dumps(normalized, sort_keys=True)


def update_manifest(extra_payload: dict) -> dict:
    manifest_path = CHECKPOINT_DIR / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    manifest.update(extra_payload)
    write_json_atomic(manifest_path, manifest)
    return manifest


def create_resume_bundle(bundle_name: str = DEFAULT_BUNDLE_NAME, include_data: bool = True) -> Path:
    bundle_path = EXPORT_DIR / bundle_name
    if bundle_path.exists():
        bundle_path.unlink()

    paths_to_pack = []
    if include_data:
        paths_to_pack.append(DATA_DIR)
    paths_to_pack.extend([OUTPUT_ROOT, SUBMISSION_DIR])

    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
        for root_path in paths_to_pack:
            if not root_path.exists():
                continue
            for nested in root_path.rglob("*"):
                if nested.is_dir():
                    continue
                relative_path = nested.relative_to(WORKSPACE_ROOT)
                zip_file.write(nested, arcname=str(relative_path))

    return bundle_path


def checkpoint_housekeeping(stage_name: str, *, refresh_bundle: bool = False, include_data_in_bundle: bool = False) -> dict:
    payload = {"last_checkpoint_stage": stage_name}
    if refresh_bundle:
        bundle_path = create_resume_bundle(include_data=include_data_in_bundle)
        payload["resume_bundle_path"] = str(bundle_path)
        payload["resume_bundle_size_mb"] = round(bundle_path.stat().st_size / (1024 ** 2), 3)
    update_manifest(payload)
    return payload

## 6. Strategy-specific helpers

In [ ]:
FEATURE_SET_NAMES = ["basic", "fft", "all"]


def normalize_candidate(params: dict) -> dict:
    normalized = {key: normalize_value(value) for key, value in params.items()}
    normalized["feature__set"] = str(normalized["feature__set"])
    normalized["feature__raw_pca"] = int(normalized["feature__raw_pca"])
    normalized["scale__method"] = str(normalized["scale__method"])
    normalized["model__family"] = str(normalized["model__family"])
    if normalized["model__family"] == "knn":
        normalized["model__n_neighbors"] = int(normalized["model__n_neighbors"])
        normalized["model__metric"] = str(normalized["model__metric"])
        normalized["model__C"] = None
        normalized["model__gamma"] = None
    else:
        normalized["model__n_neighbors"] = None
        normalized["model__metric"] = None
        normalized["model__C"] = float(normalized["model__C"])
        normalized["model__gamma"] = float(normalized["model__gamma"])
    return normalized


def candidate_record(candidate_id: str, params: dict, source: str) -> dict:
    params = normalize_candidate(params)
    return {
        "candidate_id": candidate_id,
        "source": source,
        "params": params,
        "signature": candidate_signature(params),
    }


def candidate_to_row(candidate: dict) -> dict:
    params = candidate["params"]
    return {
        "candidate_id": candidate["candidate_id"],
        "source": candidate["source"],
        "signature": candidate["signature"],
        "feature__set": params["feature__set"],
        "feature__raw_pca": params["feature__raw_pca"],
        "scale__method": params["scale__method"],
        "model__family": params["model__family"],
        "model__n_neighbors": params["model__n_neighbors"],
        "model__metric": params["model__metric"],
        "model__C": params["model__C"],
        "model__gamma": params["model__gamma"],
        "params_json": json.dumps(params, sort_keys=True),
    }


def choose_scaler(name: str):
    if name == "standard":
        return StandardScaler()
    if name == "robust":
        return RobustScaler()
    raise ValueError(f"Unknown scaler: {name}")


def _safe_divide(num: np.ndarray, den: np.ndarray) -> np.ndarray:
    return np.divide(num, den, out=np.zeros_like(num, dtype=np.float64), where=np.abs(den) > 1e-12)


def build_feature_bank(X_input: np.ndarray) -> dict[str, np.ndarray]:
    X = X_input.astype(np.float64)
    eps = 1e-12
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    minimum = X.min(axis=1)
    maximum = X.max(axis=1)
    median = np.median(X, axis=1)
    q25 = np.quantile(X, 0.25, axis=1)
    q75 = np.quantile(X, 0.75, axis=1)
    iqr = q75 - q25
    rms = np.sqrt(np.mean(X ** 2, axis=1))
    abs_mean = np.mean(np.abs(X), axis=1)
    max_abs = np.max(np.abs(X), axis=1)
    peak_to_peak = maximum - minimum
    centered = X - mean[:, None]
    skew = _safe_divide(np.mean(centered ** 3, axis=1), (std ** 3) + eps)
    kurt = _safe_divide(np.mean(centered ** 4, axis=1), (std ** 4) + eps)
    zero_cross = np.mean((X[:, 1:] * X[:, :-1]) < 0, axis=1)
    energy = np.mean(X ** 2, axis=1)
    diffs = np.diff(X, axis=1)
    diff_energy = np.mean(diffs ** 2, axis=1)
    corr1 = _safe_divide(np.sum(centered[:, 1:] * centered[:, :-1], axis=1), np.sum(centered[:, :-1] ** 2, axis=1) + eps)
    corr2 = _safe_divide(np.sum(centered[:, 2:] * centered[:, :-2], axis=1), np.sum(centered[:, :-2] ** 2, axis=1) + eps)
    corr4 = _safe_divide(np.sum(centered[:, 4:] * centered[:, :-4], axis=1), np.sum(centered[:, :-4] ** 2, axis=1) + eps)

    basic = np.column_stack(
        [
            mean, std, minimum, maximum, median, q25, q75, iqr, rms,
            abs_mean, max_abs, peak_to_peak, skew, kurt, zero_cross,
            energy, diff_energy, corr1, corr2, corr4,
        ]
    )

    fft_mag = np.abs(np.fft.rfft(X, axis=1))
    fft_power = fft_mag ** 2
    freq_idx = np.arange(fft_mag.shape[1], dtype=np.float64)
    total_mag = fft_mag.sum(axis=1) + eps
    spectral_centroid = np.sum(fft_mag * freq_idx[None, :], axis=1) / total_mag
    dominant_idx = np.argmax(fft_mag[:, 1:], axis=1) + 1
    dominant_amp = np.max(fft_mag[:, 1:], axis=1)
    p = fft_power / (fft_power.sum(axis=1, keepdims=True) + eps)
    spectral_entropy = -np.sum(p * np.log(p + eps), axis=1)
    split_points = np.linspace(0, fft_mag.shape[1], 5, dtype=int)
    band_energy = []
    for left, right in zip(split_points[:-1], split_points[1:]):
        band_energy.append(fft_power[:, left:right].sum(axis=1))
    fft_features = np.column_stack([spectral_centroid, dominant_idx, dominant_amp, spectral_entropy, *band_energy])

    segments = np.array_split(X, 4, axis=1)
    seg_stats = []
    for seg in segments:
        seg_stats.extend(
            [
                seg.mean(axis=1),
                seg.std(axis=1),
                np.sqrt(np.mean(seg ** 2, axis=1)),
                np.max(np.abs(seg), axis=1),
            ]
        )
    segment_features = np.column_stack(seg_stats)

    return {
        "basic": basic.astype(np.float32),
        "fft": np.hstack([basic, fft_features]).astype(np.float32),
        "all": np.hstack([basic, fft_features, segment_features]).astype(np.float32),
    }


FEATURE_BANK_CACHE = {}


def get_feature_bank(cache_key: str, X_input: np.ndarray) -> dict[str, np.ndarray]:
    if cache_key not in FEATURE_BANK_CACHE:
        FEATURE_BANK_CACHE[cache_key] = build_feature_bank(X_input)
    return FEATURE_BANK_CACHE[cache_key]


def build_matrix(X_fit: np.ndarray, X_eval: np.ndarray, candidate: dict) -> tuple[np.ndarray, np.ndarray]:
    params = candidate["params"]
    fit_bank = build_feature_bank(X_fit)
    eval_bank = build_feature_bank(X_eval)
    X_fit_base = fit_bank[params["feature__set"]]
    X_eval_base = eval_bank[params["feature__set"]]
    if params["feature__raw_pca"] > 0:
        raw_scaler = StandardScaler()
        X_fit_raw_scaled = raw_scaler.fit_transform(X_fit)
        X_eval_raw_scaled = raw_scaler.transform(X_eval)
        raw_pca = PCA(n_components=min(params["feature__raw_pca"], X_fit_raw_scaled.shape[0], X_fit_raw_scaled.shape[1]), random_state=RANDOM_STATE)
        X_fit_raw = raw_pca.fit_transform(X_fit_raw_scaled).astype(np.float32)
        X_eval_raw = raw_pca.transform(X_eval_raw_scaled).astype(np.float32)
        X_fit_base = np.hstack([X_fit_base, X_fit_raw]).astype(np.float32)
        X_eval_base = np.hstack([X_eval_base, X_eval_raw]).astype(np.float32)
    scaler = choose_scaler(params["scale__method"])
    X_fit_scaled = scaler.fit_transform(X_fit_base).astype(np.float32)
    X_eval_scaled = scaler.transform(X_eval_base).astype(np.float32)
    return X_fit_scaled, X_eval_scaled


def build_model(candidate: dict, probability: bool = False):
    params = candidate["params"]
    if params["model__family"] == "knn":
        return KNeighborsClassifier(
            n_neighbors=int(params["model__n_neighbors"]),
            metric=params["model__metric"],
            weights="distance",
            algorithm="brute",
            n_jobs=KNN_N_JOBS,
        )
    return SVC(
        kernel="rbf",
        C=float(params["model__C"]),
        gamma=float(params["model__gamma"]),
        cache_size=SVM_CACHE_MB,
        probability=probability,
    )


def fit_bundle(X_fit: np.ndarray, y_fit: np.ndarray, candidate: dict, probability: bool = False) -> dict:
    X_fit_matrix, _ = build_matrix(X_fit, X_fit[: min(2, len(X_fit))], candidate)
    model = build_model(candidate, probability=probability)
    model.fit(X_fit_matrix, y_fit)
    return {"candidate": candidate, "model": model}


def predict_with_bundle(bundle: dict, X_fit_reference: np.ndarray, X_input: np.ndarray, proba: bool = False) -> np.ndarray:
    X_fit_matrix, X_eval_matrix = build_matrix(X_fit_reference, X_input, bundle["candidate"])
    if X_fit_matrix.shape[0] != len(X_fit_reference):
        raise RuntimeError("Unexpected matrix shape during prediction.")
    if proba:
        return bundle["model"].predict_proba(X_eval_matrix)
    return bundle["model"].predict(X_eval_matrix)


def evaluate_candidate(X_fit: np.ndarray, y_fit: np.ndarray, X_eval: np.ndarray, y_eval: np.ndarray, candidate: dict) -> tuple[float, float]:
    start = time.time()
    X_fit_matrix, X_eval_matrix = build_matrix(X_fit, X_eval, candidate)
    model = build_model(candidate, probability=False)
    model.fit(X_fit_matrix, y_fit)
    predictions = model.predict(X_eval_matrix)
    accuracy = accuracy_score(y_eval, predictions)
    elapsed = round(time.time() - start, 3)
    return float(accuracy), elapsed


def refresh_summary_from_folds(fold_path: Path, summary_path: Path, candidates: list[dict], n_splits: int, stage_name: str) -> pd.DataFrame:
    fold_df = read_dataframe(fold_path)
    if fold_df.empty:
        return pd.DataFrame()
    rows = []
    for candidate in candidates:
        candidate_df = fold_df[fold_df["candidate_id"] == candidate["candidate_id"]]
        if candidate_df["fold_idx"].nunique() < n_splits:
            continue
        rows.append(
            {
                **candidate_to_row(candidate),
                "stage": stage_name,
                "cv_mean_accuracy": float(candidate_df["fold_accuracy"].mean()),
                "cv_std_accuracy": float(candidate_df["fold_accuracy"].std(ddof=0)),
                "total_fit_seconds": float(candidate_df["fit_seconds"].sum()),
            }
        )
    summary_df = pd.DataFrame(rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(["cv_mean_accuracy", "total_fit_seconds"], ascending=[False, True]).reset_index(drop=True)
        save_dataframe_atomic(summary_df, summary_path)
    return summary_df

## 7. Validate that the required CSV files are present

In [ ]:
missing_files = [str(path) for path in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH] if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required data files. Upload training.csv, test.csv and sample.csv first or restore a resume ZIP. "
        f"Missing: {missing_files}"
    )

print("All required CSV files are present.")

## 8. Runtime inspection and search budget

In [ ]:
CPU_COUNT = os.cpu_count() or 2
RAM_GB = None if psutil is None else round(psutil.virtual_memory().total / (1024 ** 3), 2)
SVM_CACHE_MB = 1024 if (RAM_GB is not None and RAM_GB >= 20) else 512
KNN_N_JOBS = max(1, CPU_COUNT - 1)

SEARCH_PRESETS = {
    "balanced": {"stage1_max_candidates": 54, "stage2_top_k": 12, "stage2_cv": 3},
    "aggressive": {"stage1_max_candidates": 84, "stage2_top_k": 18, "stage2_cv": 5},
}

SEARCH_PROFILE = "aggressive"
PROFILE = SEARCH_PRESETS[SEARCH_PROFILE]

RUN_STAGE_1 = True
RUN_STAGE_2 = True
TRAIN_FINAL_MODEL = True

print(
    {
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
        "cpu_count": CPU_COUNT,
        "ram_gb": RAM_GB,
        "search_profile": SEARCH_PROFILE,
    }
)

## 9. Load the challenge data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

feature_names = [column for column in train_df.columns if column not in {"id", "class"}]
train_ids = train_df["id"].to_numpy()
test_ids = test_df["id"].to_numpy()

X_full = train_df[feature_names].astype(np.float32).to_numpy()
y_full = train_df["class"].astype(np.int8).to_numpy()
X_test_full = test_df[feature_names].astype(np.float32).to_numpy()

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Class balance:", train_df["class"].value_counts().sort_index().to_dict())
print("Missing values in training:", int(train_df.isna().sum().sum()))
print("Duplicated rows in training:", int(train_df.duplicated().sum()))

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=VALID_SIZE,
    stratify=y_full,
    random_state=RANDOM_STATE,
)

print("Train split:", X_train.shape, y_train.shape)
print("Validation split:", X_valid.shape, y_valid.shape)

## 10. Stage 1

In [ ]:
candidate_list = []
next_index = 0
for feature_set in FEATURE_SET_NAMES:
    for raw_pca in [0, 16, 32]:
        for scale_method in ["standard", "robust"]:
            for k_value, metric_value in [(3, "manhattan"), (5, "manhattan"), (7, "euclidean")]:
                candidate_list.append(
                    candidate_record(
                        f"stage1_{next_index:03d}",
                        {
                            "feature__set": feature_set,
                            "feature__raw_pca": raw_pca,
                            "scale__method": scale_method,
                            "model__family": "knn",
                            "model__n_neighbors": k_value,
                            "model__metric": metric_value,
                            "model__C": None,
                            "model__gamma": None,
                        },
                        "stage1_manual_bank",
                    )
                )
                next_index += 1
            for c_value, gamma_value in [(2.0, 0.005), (3.0, 0.01), (6.0, 0.01)]:
                candidate_list.append(
                    candidate_record(
                        f"stage1_{next_index:03d}",
                        {
                            "feature__set": feature_set,
                            "feature__raw_pca": raw_pca,
                            "scale__method": scale_method,
                            "model__family": "svm",
                            "model__n_neighbors": None,
                            "model__metric": None,
                            "model__C": c_value,
                            "model__gamma": gamma_value,
                        },
                        "stage1_manual_bank",
                    )
                )
                next_index += 1

rng = np.random.default_rng(RANDOM_STATE)
if len(candidate_list) > PROFILE["stage1_max_candidates"]:
    sampled_indices = sorted(rng.choice(len(candidate_list), size=PROFILE["stage1_max_candidates"], replace=False))
    stage1_candidates = [candidate_list[idx] for idx in sampled_indices]
else:
    stage1_candidates = candidate_list

write_json_atomic(CHECKPOINT_DIR / "stage1_candidates.json", stage1_candidates)
stage1_results_path = CHECKPOINT_DIR / "stage1_holdout_results.csv"
stage1_results = read_dataframe(stage1_results_path)
completed_stage1 = set(stage1_results["candidate_id"]) if not stage1_results.empty else set()

X_stage1_fit, X_stage1_eval, y_stage1_fit, y_stage1_eval = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

print("Stage 1 candidates:", len(stage1_candidates))
print("Stage 1 completed:", len(completed_stage1))

if RUN_STAGE_1:
    for candidate in tqdm([c for c in stage1_candidates if c["candidate_id"] not in completed_stage1], desc="Stage 1 signal feature screening"):
        accuracy, elapsed = evaluate_candidate(X_stage1_fit, y_stage1_fit, X_stage1_eval, y_stage1_eval, candidate)
        row = {
            **candidate_to_row(candidate),
            "stage": "stage1_holdout",
            "holdout_accuracy": accuracy,
            "fit_seconds": elapsed,
        }
        stage1_results = pd.concat([stage1_results, pd.DataFrame([row])], ignore_index=True) if not stage1_results.empty else pd.DataFrame([row])
        stage1_results = stage1_results.sort_values(["holdout_accuracy", "fit_seconds"], ascending=[False, True]).reset_index(drop=True)
        save_dataframe_atomic(stage1_results, stage1_results_path)
        update_manifest(
            {
                "stage1_completed_candidates": int(stage1_results["candidate_id"].nunique()),
                "stage1_total_candidates": len(stage1_candidates),
                "stage1_best_holdout_accuracy": float(stage1_results["holdout_accuracy"].max()),
            }
        )
        gc.collect()

stage1_results = read_dataframe(stage1_results_path)
if not stage1_results.empty:
    display(stage1_results.head(15))
    checkpoint_housekeeping("stage1_complete", refresh_bundle=True, include_data_in_bundle=False)

## 11. Stage 2

In [ ]:
stage2_fold_path = CHECKPOINT_DIR / "stage2_cv_fold_results.csv"
stage2_summary_path = CHECKPOINT_DIR / "stage2_cv_summary.csv"

if stage1_results.empty:
    raise RuntimeError("Stage 1 produced no results.")

stage2_candidates = [
    candidate_record(row["candidate_id"], json.loads(row["params_json"]), "stage2_shortlist_from_stage1")
    for row in (
        stage1_results.sort_values(["holdout_accuracy", "fit_seconds"], ascending=[False, True])
        .drop_duplicates("signature")
        .head(PROFILE["stage2_top_k"])
        .to_dict(orient="records")
    )
]

stage2_cv = StratifiedKFold(n_splits=PROFILE["stage2_cv"], shuffle=True, random_state=RANDOM_STATE)
stage2_splits = list(stage2_cv.split(X_train, y_train))
fold_df = read_dataframe(stage2_fold_path)
completed_pairs = set(zip(fold_df["candidate_id"].astype(str), fold_df["fold_idx"].astype(int))) if not fold_df.empty else set()

print("Stage 2 candidates:", len(stage2_candidates))
print("Stage 2 completed fold-pairs:", len(completed_pairs))

if RUN_STAGE_2:
    for fold_idx, (fit_idx, eval_idx) in enumerate(tqdm(stage2_splits, desc="Stage 2 folds")):
        pending = [candidate for candidate in stage2_candidates if (candidate["candidate_id"], fold_idx) not in completed_pairs]
        for candidate in tqdm(pending, desc=f"Stage 2 fold {fold_idx}", leave=False):
            accuracy, elapsed = evaluate_candidate(
                X_train[fit_idx],
                y_train[fit_idx],
                X_train[eval_idx],
                y_train[eval_idx],
                candidate,
            )
            row = {
                **candidate_to_row(candidate),
                "stage": "stage2_cv",
                "fold_idx": fold_idx,
                "fold_accuracy": accuracy,
                "fit_seconds": elapsed,
            }
            fold_df = pd.concat([fold_df, pd.DataFrame([row])], ignore_index=True) if not fold_df.empty else pd.DataFrame([row])
            save_dataframe_atomic(fold_df, stage2_fold_path)
            completed_pairs.add((candidate["candidate_id"], fold_idx))
            gc.collect()

        stage2_summary = refresh_summary_from_folds(stage2_fold_path, stage2_summary_path, stage2_candidates, PROFILE["stage2_cv"], "stage2_cv")
        if not stage2_summary.empty:
            update_manifest(
                {
                    "stage2_completed_candidates": int(stage2_summary["candidate_id"].nunique()),
                    "stage2_total_candidates": len(stage2_candidates),
                    "stage2_best_cv_accuracy": float(stage2_summary["cv_mean_accuracy"].max()),
                }
            )

stage2_summary = read_dataframe(stage2_summary_path)
if not stage2_summary.empty:
    display(stage2_summary.head(15))
    checkpoint_housekeeping("stage2_complete", refresh_bundle=True, include_data_in_bundle=False)

## 12. Stage 3

In [ ]:
stage3_summary = stage2_summary.copy() if not stage2_summary.empty else pd.DataFrame()
if not stage3_summary.empty:
    display(stage3_summary.head(12))
    checkpoint_housekeeping("stage3_proxy_complete", refresh_bundle=True, include_data_in_bundle=False)

## 13. Diagnostic plots

In [ ]:
if not stage1_results.empty:
    plt.figure(figsize=(12, 6))
    stage1_plot = stage1_results.groupby(["feature__set", "model__family"], as_index=False)["holdout_accuracy"].max()
    sns.barplot(data=stage1_plot, x="feature__set", y="holdout_accuracy", hue="model__family")
    plt.title("Best holdout accuracy by feature set and model family")
    save_current_figure("stage1_feature_family.png")
    display(stage1_plot)

if not stage2_summary.empty:
    plt.figure(figsize=(12, 6))
    top_df = stage2_summary.head(15).copy()
    sns.barplot(data=top_df, x="candidate_id", y="cv_mean_accuracy", hue="model__family")
    plt.xticks(rotation=75, ha="right")
    plt.title("Top signal feature candidates")
    save_current_figure("top_candidates.png")

## 14. Final model, OOF artifacts and submission

In [ ]:
final_summary_df = stage2_summary if not stage2_summary.empty else stage1_results
if final_summary_df.empty:
    raise RuntimeError("No final candidate table is available.")

best_row = final_summary_df.iloc[0].to_dict()
best_candidate = candidate_record(best_row["candidate_id"], json.loads(best_row["params_json"]), "final_selection")

X_fit_matrix, X_eval_matrix = build_matrix(X_train, X_valid, best_candidate)
final_model = build_model(best_candidate, probability=False)
final_model.fit(X_fit_matrix, y_train)
valid_predictions = final_model.predict(X_eval_matrix)
validation_accuracy = float(accuracy_score(y_valid, valid_predictions))

cm = confusion_matrix(y_valid, valid_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", colorbar=False)
plt.title(f"Validation confusion matrix - accuracy={validation_accuracy:.4f}")
save_current_figure("validation_confusion_matrix.png")

oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_prob_1 = np.zeros(len(X_full), dtype=np.float32)
test_prob_1 = np.zeros(len(X_test_full), dtype=np.float32)
fold_records = []

for fold_idx, (fit_idx, eval_idx) in enumerate(tqdm(oof_cv.split(X_full, y_full), desc="OOF for stacking")):
    X_fold_fit, X_fold_eval = build_matrix(X_full[fit_idx], X_full[eval_idx], best_candidate)
    X_fold_fit_again, X_fold_test = build_matrix(X_full[fit_idx], X_test_full, best_candidate)
    if X_fold_fit.shape != X_fold_fit_again.shape:
        raise RuntimeError("Unexpected feature matrix mismatch.")
    fold_model = build_model(best_candidate, probability=(best_candidate["params"]["model__family"] == "svm"))
    fold_model.fit(X_fold_fit, y_full[fit_idx])
    if best_candidate["params"]["model__family"] == "knn":
        oof_proba = fold_model.predict_proba(X_fold_eval)[:, 1]
        test_proba = fold_model.predict_proba(X_fold_test)[:, 1]
    else:
        oof_proba = fold_model.predict_proba(X_fold_eval)[:, 1]
        test_proba = fold_model.predict_proba(X_fold_test)[:, 1]
    oof_prob_1[eval_idx] = oof_proba.astype(np.float32)
    test_prob_1 += test_proba.astype(np.float32) / oof_cv.n_splits
    fold_records.append(
        {
            "fold_idx": fold_idx,
            "fold_accuracy": float(accuracy_score(y_full[eval_idx], (oof_proba >= 0.5).astype(int))),
        }
    )
    gc.collect()

X_full_matrix, X_test_matrix = build_matrix(X_full, X_test_full, best_candidate)
final_model_full = build_model(best_candidate, probability=False)
final_model_full.fit(X_full_matrix, y_full)
final_test_pred = final_model_full.predict(X_test_matrix).astype(int)

oof_df = pd.DataFrame(
    {
        "id": train_ids,
        "y_true": y_full.astype(int),
        "prob_1": oof_prob_1,
        "pred": (oof_prob_1 >= 0.5).astype(int),
        "source_model": NOTEBOOK_SLUG,
    }
)
test_prob_df = pd.DataFrame(
    {
        "id": test_ids,
        "prob_1": test_prob_1,
        "pred": (test_prob_1 >= 0.5).astype(int),
        "source_model": NOTEBOOK_SLUG,
    }
)
submission_df = sample_df.copy()
submission_df["class"] = final_test_pred.astype(int)

oof_path = PERSIST_ROOT / "oof_probabilities.csv"
test_prob_path = PERSIST_ROOT / "test_probabilities.csv"
fold_summary_path = PERSIST_ROOT / "oof_fold_summary.csv"
submission_path = SUBMISSION_DIR / "challenge_10_signal_features_colab_ultra_submission.csv"
summary_path = PERSIST_ROOT / "summary.json"

save_dataframe_atomic(oof_df, oof_path)
save_dataframe_atomic(test_prob_df, test_prob_path)
save_dataframe_atomic(pd.DataFrame(fold_records), fold_summary_path)
submission_df.to_csv(submission_path, index=False)

summary_payload = {
    "model_name": "Signal feature engineering model",
    "model_key": "signal_features",
    "notebook_slug": NOTEBOOK_SLUG,
    "strategy": "colab_ultra_signal_features",
    "search_profile": SEARCH_PROFILE,
    "best_stage": str(best_row.get("stage", "unknown")),
    "best_params": best_candidate["params"],
    "validation_accuracy": validation_accuracy,
    "validation_confusion_matrix": cm.tolist(),
    "oof_accuracy": float(accuracy_score(y_full, (oof_prob_1 >= 0.5).astype(int))),
    "oof_path": str(oof_path),
    "test_probability_path": str(test_prob_path),
    "submission_path": str(submission_path),
    "workspace_root": str(WORKSPACE_ROOT),
    "persist_root": str(PERSIST_ROOT),
}
write_json_atomic(summary_path, summary_payload)
checkpoint_housekeeping("final_model_complete", refresh_bundle=True, include_data_in_bundle=False)

print("Best params:", json.dumps(best_candidate["params"], indent=2))
print("Validation accuracy:", validation_accuracy)
print("OOF accuracy:", summary_payload["oof_accuracy"])
print("Submission path:", submission_path)

## 15. Optional: export and download a manual resume bundle

In [ ]:
INCLUDE_DATA_IN_MANUAL_BUNDLE = True
DOWNLOAD_BUNDLE_NOW = False

bundle_path = create_resume_bundle(include_data=INCLUDE_DATA_IN_MANUAL_BUNDLE)
print("Resume bundle saved to:", bundle_path)

if DOWNLOAD_BUNDLE_NOW and IN_COLAB:
    files.download(str(bundle_path))

## Notes

Si esta notebook supera a los pipelines con `raw features`, la evidencia es fuerte:

- el orden de `V1 ... V200` si contiene estructura util de señal
- el challenge se beneficia mas de `feature engineering` que de seguir moviendo hiperparametros